# H1c 力のモーメント：握る力と重さ

**ねらい**：作品の芯の 2 つの量を、自分で測る。
- **握る力**：指のモーターの力の上限 τ [N·m] を決めると、指が箱を挟む力 F [N] が決まる（力のモーメント F × r ＝ τ）
- **重さ**：箱を持って止めると、肩のモーターが出す力が増える。その増え方から重さが出る（Δτ ＝ m g L）

そして、**持てる条件 2μF ≥ mg**（2 本の指の摩擦で、箱の重さを支える）を SO-101 で確かめる。
理論どおりになる所と、ならない所（関節の摩擦、掴みに行く動き）を見分ける。

## 1. 準備

In [ ]:
import os, time
if "google.colab" in str(get_ipython()):
    !apt-get -qq install -y libosmesa6 fonts-noto-cjk > /dev/null
    !pip -q install genesis-world==1.4.3
    if not os.path.exists("SO-ARM100"):
        !git clone -q https://github.com/TheRobotStudio/SO-ARM100.git && cd SO-ARM100 && git checkout -q a758567
    MODEL = "/content/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
else:   # 手元の Docker（答え合わせ用）
    MODEL = "/work/third_party/SO-ARM100/Simulation/SO101/so101_new_calib.xml"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
import numpy as np, matplotlib.pyplot as plt, genesis as gs
# グラフの日本語：Colab は matplotlib を先に読み込んでいるので、入れたフォントを直接登録する（名前だけ指定しても見つからない）
from matplotlib import font_manager
for _f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
    if os.path.exists(_f):
        font_manager.fontManager.addfont(_f)
plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]

## 2. 道具と場面
H1b の回転の関数に加えて、`jaws()`：指が箱を押す力を、固定側の指・動く側の指に分けて読む（`get_contacts`：触れている所の一覧と力）。
挟む力は **2 本の小さいほう**（片側の指が触れただけでは、箱は逃げるので挟んでいない）。

箱は 3 cm 角、床の上、根元から 22 cm。重さは `set_mass` で変える。

In [ ]:
import math
OFF = np.array([-0.0079, -0.000218, -0.0981]); FOLD = np.array([0.0, -1.7, 1.6, 1.2, 0.0, 0.0])
OPEN, CLOSE, PRE, JAW = 1.2, -0.17, 0.5, 0.019    # 指の角度（開く／閉じきる／速く閉じる 1 段目の止め所）、目印 → 箱の中心
def rot(q):
    w, x, y, z = q
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])
def quat_of(R):
    t = np.trace(R)
    if t > 0:
        S = math.sqrt(t + 1.0) * 2; q = [0.25 * S, (R[2, 1] - R[1, 2]) / S, (R[0, 2] - R[2, 0]) / S, (R[1, 0] - R[0, 1]) / S]
    elif R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
        S = math.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2]) * 2; q = [(R[2, 1] - R[1, 2]) / S, 0.25 * S, (R[0, 1] + R[1, 0]) / S, (R[0, 2] + R[2, 0]) / S]
    elif R[1, 1] > R[2, 2]:
        S = math.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2]) * 2; q = [(R[0, 2] - R[2, 0]) / S, (R[0, 1] + R[1, 0]) / S, 0.25 * S, (R[1, 2] + R[2, 1]) / S]
    else:
        S = math.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1]) * 2; q = [(R[1, 0] - R[0, 1]) / S, (R[0, 2] + R[2, 0]) / S, (R[1, 2] + R[2, 1]) / S, 0.25 * S]
    q = np.array(q); return q / np.linalg.norm(q)
def Ry(a): return np.array([[math.cos(a), 0, math.sin(a)], [0, 1, 0], [-math.sin(a), 0, math.cos(a)]])

gs.init(backend=gs.cpu, logging_level="warning")
sc = gs.Scene(sim_options=gs.options.SimOptions(dt=0.01, substeps=10),
              rigid_options=gs.options.RigidOptions(box_box_detection=True, friction_cone=gs.friction_cone.elliptic, impratio=10.0),
              show_viewer=False)
sc.add_entity(gs.morphs.Plane())
A = sc.add_entity(gs.morphs.MJCF(file=MODEL, decompose_robot_error_threshold=0.0), material=gs.materials.Rigid(friction=1.0))
RB = 0.22
box = sc.add_entity(gs.morphs.Box(size=(0.03,) * 3, pos=(RB, 0, 0.015)), material=gs.materials.Rigid(friction=1.0, rho=1000.0),
                    surface=gs.surfaces.Default(color=(0.72, 0.55, 0.36)))
cam = sc.add_camera(res=(768, 432), pos=(0.22, -0.30, 0.15), lookat=(0.20, 0.0, 0.03), fov=45, GUI=False)
t0 = time.time(); sc.build(); print(f"build {time.time() - t0:.0f} s")
g, upper = A.get_link("gripper"), A.get_link("upper_arm")
FIXED, MOVING = g.idx, A.get_link("moving_jaw_so101_v1").idx
A.set_dofs_position(np.zeros(6)); Rd = Ry(math.pi / 2) @ rot(g.get_quat().cpu().numpy())   # 真下向き（H1b）
FL0 = A.get_dofs_frictionloss().cpu().numpy()

def ik(p):
    return A.inverse_kinematics(link=g, pos=p - Rd @ OFF, quat=quat_of(Rd), init_qpos=A.get_dofs_position(), max_solver_iters=100).cpu().numpy()
def tip():
    return g.get_pos().cpu().numpy() + rot(g.get_quat().cpu().numpy()) @ OFF
def jaws():
    """(固定側の指が押す力, 動く側の指が押す力) [N]"""
    c = A.get_contacts(with_entity=box)
    la, lb = c["link_a"].cpu().numpy(), c["link_b"].cpu().numpy()
    f = np.linalg.norm(c["force_a"].cpu().numpy(), axis=-1)
    if "valid_mask" in c: f = f * c["valid_mask"].cpu().numpy()      # 並列の環境のときだけ付く（空きの欄の印）
    return float((f * ((la == FIXED) | (lb == FIXED))).sum()), float((f * ((la == MOVING) | (lb == MOVING))).sum())
def squeeze():
    return min(jaws())
def tilt():
    q = box.get_quat().cpu().numpy(); return math.degrees(math.acos(max(-1.0, min(1.0, 1 - 2 * (q[1] ** 2 + q[2] ** 2)))))
def set_grip(tau):
    """指のモーターの力の上限 = 握る力の設定 [N·m]（ほかの関節は 3.35）"""
    lim = np.full(6, 3.35); lim[5] = tau; A.set_dofs_force_range(-lim, lim)

cur = [None]
def go(p, grip, n, every=None):
    """目印を p へ、真下向きのまま n step かけて S 字で動かす（H1b の move と同じ）。指の角度は grip。"""
    start = cur[0]
    for k in range(n):
        s = min(1.0, (k + 1) / (0.7 * n)); s = 3 * s * s - 2 * s ** 3
        if k % 5 == 0 or k == n - 1:
            q = ik(start + (p - start) * s); q[5] = grip; A.control_dofs_position(q)
        sc.step()
        if every: every()
    cur[0] = np.array(p, float)
def start(mass, frictionloss=None):
    """初めの状態に戻し、箱の重さを mass [kg] に。指を開いて、箱の上 8 cm に置く"""
    sc.reset(); box.set_mass(mass); set_grip(3.35)
    fl = FL0.copy()
    if frictionloss is not None: fl[5] = frictionloss
    A.set_dofs_frictionloss(fl)
    cur[0] = np.array([RB - JAW, 0, 0.08]); q = ik(cur[0]); q[5] = OPEN
    A.set_dofs_position(q); A.control_dofs_position(q)
    for _ in range(30): sc.step()
def grasp(tau, mass, frictionloss=None):
    """下ろして、指を 2 段階で閉じる（速く 1 段目まで → 力の上限を tau に絞って閉じきる。作品の close_gently）"""
    start(mass, frictionloss)
    go([RB - JAW, 0, 0.02], OPEN, 100)
    go(cur[0], PRE, 40)
    set_grip(tau); go(cur[0], CLOSE, 200)
def show():
    sc.visualizer.update(force=True); plt.figure(figsize=(8, 4.5)); plt.imshow(cam.render(rgb=True)[0]); plt.axis("off"); plt.show()

grasp(0.6, 0.15); print("0.6 N·m で 150 g を挟んだ：固定側 %.2f N、動く側 %.2f N" % jaws()); show()

## 3. 握る力の設定 τ → 挟む力 F

指は「閉じきれ」（角度 −0.17）と指示されている。箱に当たって止まると、モーターは **力の上限 τ いっぱい** で押し続ける。
指の関節の軸から、箱に触れる所までの長さを r とすると、**F × r ＝ τ**（力のモーメント。てこと同じ）。

**予想**：τ を 2 倍にすると F は？ F ÷ τ はいくつ？

In [ ]:
TAUS = [0.3, 0.4, 0.6, 0.8]
F = []
for tau in TAUS:
    grasp(tau, 0.15); ff, fm = jaws(); F.append(min(ff, fm))
    print(f"τ {tau:.2f} N·m   固定側 {ff:5.2f} N   動く側 {fm:5.2f} N   挟む力 {min(ff, fm):5.2f} N   F ÷ τ {min(ff, fm) / tau:5.1f} N/(N·m)")
k = float(np.dot(TAUS, F) / np.dot(TAUS, TAUS))     # 原点を通る直線 F ＝ k τ の傾き
print(f"傾き {k:.1f} N/(N·m)  →  r ＝ 1 ÷ 傾き ＝ {100 / k:.1f} cm（指の関節の軸から、箱に触れる所まで）")
plt.figure(figsize=(5, 3.5)); plt.plot(TAUS, F, "o-"); plt.xlabel("握る力の設定 τ [N·m]"); plt.ylabel("挟む力 F [N]"); plt.show()

**期待される出力**（手元の実測）：F ÷ τ は 13〜15、傾き 13.6 N/(N·m)、r ＝ 7.4 cm。固定側と動く側の力はほぼ同じ（箱が止まっているので、左右の力が釣り合う）。

**作品では**：この傾き（作品では 14.5 N/(N·m) と測った）を使って、「箱の強さ（段ボール 15 N）の 7 割を超えない τ」を決めている。

## ✏️ 変えてみる 1：弱い τ
**予想**：τ を 0.1、0.2 にすると、挟む力は？

In [ ]:
for tau in [0.1, 0.2]:
    grasp(tau, 0.15); ff, fm = jaws()
    print(f"τ {tau:.2f} N·m   固定側 {ff:5.2f} N   動く側 {fm:5.2f} N   挟む力 {min(ff, fm):5.2f} N")

**期待される出力**：τ 0.1 では両方 0 N、τ 0.2 では動く側だけ 1.7 N（固定側は 0）。挟む力は、どちらも 0。

**なぜ**：指を開いて下ろした所では、固定側の指と箱のあいだに数 mm のすき間がある。動く側の指は、箱を床の上で押して滑らせ、固定側の指に当てる必要がある。
弱い τ では、床の摩擦と、指の関節自身の摩擦・減衰に負けて、2 秒では挟みきれない（4 で、掴めるかどうかに効く）。

## 4. 持てる条件 2μF ≥ mg を確かめる
2 本の指がそれぞれ F で挟むと、摩擦で支えられるのは最大 μF × 2。これが箱の重さ mg 以上なら落ちない。
F ＝ 傾き × τ なので、**落ちない最小の τ ＝ m g ÷（2 μ × 傾き）**。μ は 1.0（指も箱も 1.0）。

確かめ方：まず強い力（0.6 N·m）で確実に掴んで 6 cm 持ち上げる。**持ったまま τ を 0.005 ずつ下げていき**、箱が落ちた τ を記録する。
（掴みに行く動きを混ぜないため。混ぜると 5 のように別の理由で決まってしまう）

In [ ]:
def drop_tau(mass, frictionloss=None, tau0=0.3, rate=0.005):
    grasp(0.6, mass, frictionloss); go([RB - JAW, 0, 0.06], CLOSE, 100); go(cur[0], CLOSE, 50)
    z0, tau = float(box.get_pos().cpu().numpy()[2]), tau0
    while tau > 0:
        set_grip(tau)
        for _ in range(20): sc.step()
        if float(box.get_pos().cpu().numpy()[2]) < z0 - 0.005:
            return tau
        tau = round(tau - rate, 4)
    return 0.0

def table(frictionloss=None):
    for m in [0.03, 0.15, 0.30]:
        theory = m * 9.81 / (2 * 1.0 * k)
        print(f"{m * 1000:4.0f} g   落ちた τ {drop_tau(m, frictionloss):.3f} N·m   理論 {theory:.3f} N·m")

print(f"指の関節の摩擦 {FL0[5]:.3f} N·m（モデルの値）"); table()

**期待される出力**（手元の実測）
```
  30 g   落ちた τ 0.000 N·m   理論 0.011 N·m
 150 g   落ちた τ 0.040 N·m   理論 0.054 N·m
 300 g   落ちた τ 0.110 N·m   理論 0.108 N·m
```
300 g は理論とほぼ同じ。軽い箱ほど、理論より弱い力でも落ちない（30 g は 0 まで下げても落ちない）。

## ✏️ 変えてみる 2：指の関節の摩擦
モデルのファイルには、指の関節そのものの摩擦（`frictionloss`、0.052 N·m）が書いてある。モーターの力を抜いても、この分は指が開くのを止める。
**予想**：0 にすると、軽い箱の結果はどう変わる？

In [ ]:
print("指の関節の摩擦 0"); table(frictionloss=0.0)

**期待される出力**：30 g 0.005、150 g 0.050、300 g 0.100 N·m。3 つとも理論（0.011 / 0.054 / 0.108）に近づく。

## 5. 掴みに行くと、もっと強い力が要る
今度は作品と同じく、**開いた指を下ろして、τ で閉じて、持ち上げる**。持ち上がった最小の τ を、4 の「落ちた τ」と比べる。
**予想**：4 と同じになる？

In [ ]:
def lift_ok(tau, mass):
    grasp(tau, mass); go([RB - JAW, 0, 0.04], CLOSE, 60); go(cur[0], CLOSE, 30)
    z = float(box.get_pos().cpu().numpy()[2])
    return "持てた" if z > 0.025 and tilt() < 10 else ("傾いた" if z > 0.025 else "  --  ")
TRY = [0.15, 0.2, 0.25, 0.3]
print("箱 \\ τ   " + "".join(f"{x:>8.2f}" for x in TRY))
for m in [0.03, 0.30]:
    print(f"{m * 1000:4.0f} g   " + "".join(f"{lift_ok(x, m):>6s}" for x in TRY))

**期待される出力**：30 g は 0.20 から、300 g は 0.25 から持てる（300 g・0.30 は傾いてぶら下がる）。4 の落ちた τ（0.000 / 0.110）より、ずっと強い。

**作品では**：試し掴みを 0.2 N·m から始め、持ち上がった τ（力加減）に **余裕（2 倍）** を足して運ぶ。掴みに行くときと、運ぶ途中に揺れるときは、持っているあいだの理論より強い力が要るから。

## 6. 重さを測る：肩の力の増え方
腕を止めて箱を持つと、肩のモーター（`shoulder_lift`、1 番）は、腕の重さ ＋ 箱の重さを支える力を出す。
**同じ姿勢で**、持っているときと、何も持たないときの差 Δτ を取ると、箱の分だけが残る：**Δτ ＝ m g L**（L：肩の軸から箱までの水平の長さ）。

In [ ]:
def shoulder(mass, z=0.06, empty=False, moving=False):
    """箱を持って（empty なら持たずに）高さ z で止め、肩の力の 1 秒平均と、L を返す。moving：測る間も上下に揺らす"""
    grasp(0.6, mass)
    if empty:
        box.set_pos(np.array([1.5, 1.5, 0.015]))           # 箱を遠くへ（何も持たない）
    go([RB - JAW, 0, z], CLOSE, 100)
    v = []
    for k_ in range(100):
        if moving:
            q = ik(np.array([RB - JAW, 0, z + 0.02 * math.sin(k_ / 25 * math.pi)])); q[5] = CLOSE; A.control_dofs_position(q)
        sc.step(); v.append(float(A.get_dofs_control_force()[1]))
    L = float(np.linalg.norm((box.get_pos().cpu().numpy()[:2] if not empty else tip()[:2] + np.array([JAW, 0])) - upper.get_pos().cpu().numpy()[:2]))
    return float(np.mean(v)), L

tau_empty, _ = shoulder(0.15, empty=True)
print(f"何も持たない   肩の力 {tau_empty:+.3f} N·m")
for m in [0.03, 0.10, 0.15, 0.30]:
    tau_h, L = shoulder(m)
    print(f"{m * 1000:4.0f} g を持つ   肩の力 {tau_h:+.3f} N·m   Δτ {tau_empty - tau_h:.3f}   L {L * 100:.1f} cm   測った重さ {(tau_empty - tau_h) / (9.81 * L) * 1000:5.0f} g")

**期待される出力**（手元の実測）：何も持たない −0.345 N·m。測った重さは 30 → 30 g、100 → 99 g、150 → 143 g、300 → 288 g。L は 14.9 cm。

## ✏️ 変えてみる 3：基準を決め打ちにする
作品を作る途中では、何も持たないときの肩の力を、別の姿勢で測った決め打ちの値（−0.400 N·m）にしていた。
**予想**：その値を使うと、150 g は何 g と出る？

In [ ]:
tau_h, L = shoulder(0.15)
for label, base in [("同じ姿勢で測った基準", tau_empty), ("決め打ち −0.400", -0.400)]:
    print(f"{label:14s}   測った重さ {(base - tau_h) / (9.81 * L) * 1000:5.0f} g（本当は 150 g）")

**期待される出力**：同じ姿勢の基準なら 143 g、決め打ちなら 105 g（45 g も軽く出る）。作品では、箱ごとに同じ姿勢で基準を測り直す形に直した。

## ✏️ 変えてみる 4：動かしながら測る
**予想**：測っている 1 秒のあいだ、上下に揺らすと？（平均と、ばらつき）

In [ ]:
for moving in [False, True]:
    vals = []
    for _ in range(1):
        tau_h, L = shoulder(0.15, moving=moving)
        vals.append((tau_empty - tau_h) / (9.81 * L) * 1000)
    grasp(0.6, 0.15); go([RB - JAW, 0, 0.06], CLOSE, 100); v = []
    for k_ in range(100):
        if moving:
            q = ik(np.array([RB - JAW, 0, 0.06 + 0.02 * math.sin(k_ / 25 * math.pi)])); q[5] = CLOSE; A.control_dofs_position(q)
        sc.step(); v.append(float(A.get_dofs_control_force()[1]))
    v = np.array(v)
    print(f"{'揺らしながら' if moving else '止めて':6s}   1 秒平均から {vals[0]:5.0f} g   0.1 秒ごとの値のばらつき {np.std([(tau_empty - x) / (9.81 * L) * 1000 for x in v.reshape(10, 10).mean(axis=1)]):5.0f} g")

**期待される出力**：止めて測ると 140 g 前後で、0.1 秒ごとの値のばらつきは 10 g。揺らしながらだと平均は 161 g、ばらつきは 509 g（加速の分の力が混ざる）。作品では、持ち上げて止めてから 1 秒平均を取る。

## 7. 映像で確かめる
300 g を τ ＝ 0.25 で掴んで持ち上げる（8 step に 1 コマ）。**変えてみる**：`tau=` を 0.15 にして ▶ し直す。

In [ ]:
from IPython.display import HTML
import base64, imageio
def film(tau=0.25, mass=0.30, name="h1c.mp4"):
    frames = []
    grab = lambda: frames.append(cam.render(rgb=True)[0]) if len(sc_steps) % 8 == 0 else None
    sc_steps.clear()
    def every():
        sc_steps.append(1); grab()
    start(mass); go([RB - JAW, 0, 0.02], OPEN, 100, every); go(cur[0], PRE, 40, every)
    set_grip(tau); go(cur[0], CLOSE, 200, every); go([RB - JAW, 0, 0.06], CLOSE, 100, every)
    imageio.mimsave(name, frames, fps=12)
    return HTML(f'<video controls width=640 src="data:video/mp4;base64,{base64.b64encode(open(name, "rb").read()).decode()}">')
sc_steps = []
film()

## 確かめる問い
1. 挟む力 F が τ に比例するのはなぜ？ 傾きの逆数は何の長さ？
2. 4 で、軽い箱は τ を 0 近くまで下げても落ちにくかった。なぜ？
3. 作品で、重さの基準（何も持たないときの肩の力）を、箱ごとに同じ姿勢で測り直しているのはなぜ？

<details><summary>答え</summary>

1. 指は閉じきる向きに指示され、力の上限 τ いっぱいで押す。力のモーメント F × r ＝ τ なので F ＝ τ ÷ r。傾きの逆数 r は、指の関節の軸から箱に触れる所までの長さ
2. 指の関節そのものの摩擦（0.052 N·m）が、指が開くのを止める。軽い箱が要る τ（0.01）はこれより小さい。摩擦を 0 にすると、理論どおりに落ちる
3. 腕の姿勢が変わると、腕自身の重さを支える肩の力も変わる。基準を別の姿勢で測ると、その差が箱の重さに混ざる（✏️ 3）
</details>